In [1]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "qnli"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 104743
    })
    validation: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
    test: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['question', 'sentence', 'label', 'idx'],
    num_rows: 104743
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 104743
    })
    validation: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
    test: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'question': 'When did the third Digimon series begin?',
 'sentence': 'Unlike the two seasons before it and most of the seasons that followed, Digimon Tamers takes a darker and more realistic approach to its story featuring Digimon who do not reincarnate after their deaths and more complex character development in the original Japanese.',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['question'], raw_train_dataset[5]['sentence']

("When is the term 'German dialects' used in regard to the German language?",
 'When talking about the German language, the term German dialects is only used for the traditional regional varieties.')

In [11]:
raw_train_dataset[5]['label']

0

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'question': Value('string'),
 'sentence': Value('string'),
 'label': ClassLabel(names=['entailment', 'not_entailment']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["question"], example["sentence"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map: 100%|██████████| 5463/5463 [00:00<00:00, 16660.15 examples/s]


DatasetDict({
    train: Dataset({
        features: ['question', 'sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 104743
    })
    validation: Dataset({
        features: ['question', 'sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5463
    })
    test: Dataset({
        features: ['question', 'sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5463
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["question", "sentence", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 84]),
 'token_type_ids': torch.Size([32, 84]),
 'attention_mask': torch.Size([32, 84])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.7970, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [25]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

device(type='cuda', index=1)

### Load Trained Weights

In [26]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location=device)
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
from train_eval_func import eval_loop

In [28]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.9134175361522973}

# 5. Merge Most Similar Layers

In [29]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
from tqdm import tqdm

In [30]:
import os
from merge_helper import merge_bert_layers, LayerMergeTracker
from train_eval_func import train, get_primary_metric, set_lr_scheduler, iterative_cka_merge_and_train, get_recovery_epoch
from torch.optim import AdamW

In [31]:
init_metric = eval_loop(model, val_dataloader, task_name, device)[0]
init_metric

{'accuracy': 0.9134175361522973}

In [32]:
tracker = LayerMergeTracker(num_layers=12)

print("Initial state:")
print(f"  {tracker.get_mapping()}")
print(f"  Total layers: {len(tracker)}")
print()

Initial state:
  [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11]]
  Total layers: 12



In [33]:
from CKA import CKAEvaluator

In [34]:
cka_eval = CKAEvaluator(device)

In [35]:
recovery_epochs = get_recovery_epoch(task_name)
recovery_epochs

3

In [36]:
history = iterative_cka_merge_and_train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    task_name=task_name,
    device=device,
    cka_evaluator=cka_eval,
    tracker=tracker,
    init_metric=init_metric,
    num_merges=6,
    target_layers=[6, 7, 8],  # Save models with 6, 7, 8 layers
    recovery_epochs=recovery_epochs,
    recovery_lr=1e-5,
    save_dir='./weights/',
    cka_max_iter=150
)

Target Metric for qnli: accuracy
Original accuracy score: 0.9134
Recovery threshold: 0.0091

Merge Iteration: 1/6

[1/5] Computing CKA Similarity...


CKA Evaluation: 100%|██████████| 150/150 [00:18<00:00,  8.32it/s]


  Similarity Stats:
    Average: 0.877770
    Highest: 0.959301
    Lowest:  0.711328

[2/5] Merging Layers...
  Merged Layers: 8 + 9
  Layer Composition: [[0], [1], [2], [3], [4], [5], [6], [7], [8, 9], [10], [11]]
  Total Layers: 11

[3/5] Evaluating Post-Merge Performance...
  Metrics:
    ★ accuracy: 0.8953

[4/5] Recovery Training...
  Performance drop: 0.0181 (threshold: 0.0091)
  → Recovery training NEEDED
  → Training: 3 epochs, lr=1e-05


Training:   0%|          | 0/9822 [00:00<?, ?it/s]/home/abhinavl/work/Layer_Graft/layer-grafting-publication/similar_layer_merge/../train_eval_func.py:223: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  total_loss += float(loss)
Training:  33%|███▎      | 3274/9822 [09:13<14:34,  7.49it/s]

✓ Saved checkpoint (best loss: 0.4987)

<----------------- Epoch 1 ----------------->
Loss: 0.13, Training Metrics:
  accuracy: 0.9704
Validation Loss: 0.5, Validation Metrics:
  accuracy: 0.9103
Elapsed Time: 563.6775 sec


...

Training:  67%|██████▋   | 6548/9822 [24:41<09:11,  5.94it/s]   

✓ Saved checkpoint (best loss: 0.8002)

<----------------- Epoch 2 ----------------->
Loss: 0.11, Training Metrics:
  accuracy: 0.9731
Validation Loss: 0.8, Validation Metrics:
  accuracy: 0.8896
Elapsed Time: 743.8386 sec


Training: 100%|██████████| 9822/9822 [37:37<00:00,  4.35it/s]  


<----------------- Epoch 3 ----------------->
Loss: 0.1, Training Metrics:
  accuracy: 0.9806
Validation Loss: 0.89, Validation Metrics:
  accuracy: 0.8880
Elapsed Time: 748.8397 sec

Total Training Time: 2257.33 sec
Chosen Primary Metric: accuracy
Best Metric Run: 0.8896

  → Loading best checkpoint...


  ✓ Loaded best checkpoint from epoch 2
    Best val loss: 0.8002
    Best val metrics: {'accuracy': 0.8896210873146623}
  ✓ Cleaned up temporary checkpoint

[5/5] Saving Model...
  ✓ Saved 6-layer model to ./weights/layer-6-qnli.pt

MERGE LOOP COMPLETED
Total merges: 6
Final layers: 6
Final composition: [[0], [1, 2], [3], [4], [5, 6, 7, 8, 9, 10], [11]]


In [37]:
# See what happened
print("Merge history:")
for i, iteration in enumerate(history['iterations']):
    print(f"  Iteration {iteration['iteration']}:")
    print(f"    Merged: {iteration['merged_layers']}")
    print(f"    Layers after: {iteration['num_layers_after']}")
    print(f"    Composition: {history['layer_compositions'][i]}")
    
    if history['training_stats'][i]:
        final_metrics = history['metrics_after_recovery'][i]
        print(f"    Final metrics: {final_metrics}")

Merge history:
  Iteration 1:
    Merged: (8, 9)
    Layers after: 11
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7, 8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.9121361889071938}
  Iteration 2:
    Merged: (6, 7)
    Layers after: 10
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7, 8, 9, 10], [11]]
  Iteration 3:
    Merged: (7, 8)
    Layers after: 9
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7, 8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.9044481054365733}
  Iteration 4:
    Merged: (5, 6)
    Layers after: 8
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7, 8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.9022515101592532}
  Iteration 5:
    Merged: (1, 2)
    Layers after: 7
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7, 8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.9051803038623467}
  Iteration 6:
    Merged: (4, 5)
    Layers after: 6
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7, 8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.889621087